In [1]:
import json
import random
from pathlib import Path

random.seed(42)

path = Path(r"C:\Users\shlok\projects\ddp-llm\parser\data\seed.jsonl")
examples = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

random.shuffle(examples)
fewshot = examples[:15]
eval_set = examples[15:]

print(f"fewshot: {len(fewshot)}, eval: {len(eval_set)}")

fewshot: 15, eval: 35


In [2]:
def format_user_turn(options, utterance):
    return f"Options: {options}\nUser: {utterance}"

def format_assistant_turn(label):
    # label is either a list like ["A","B"] or the string "*"
    return json.dumps(label)

SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""

def build_messages(fewshot, options, utterance):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in fewshot:
        msgs.append({"role": "user", "content": format_user_turn(ex["options"], ex["utterance"])})
        msgs.append({"role": "assistant", "content": format_assistant_turn(ex["label"])})
    msgs.append({"role": "user", "content": format_user_turn(options, utterance)})
    return msgs

# quick sanity check
print(build_messages(fewshot[:2], ["A","B","C","D"], "A and C")[-3:])

[{'role': 'user', 'content': "Options: ['A', 'B', 'C', 'D']\nUser: definitely not A or C"}, {'role': 'assistant', 'content': '["B", "D"]'}, {'role': 'user', 'content': "Options: ['A', 'B', 'C', 'D']\nUser: A and C"}]


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="cuda"
)
model.eval()
print("loaded, VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

loaded, VRAM (GB): 3.087429632


In [4]:
def generate(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy for reproducibility
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# sanity: one prediction
msgs = build_messages(fewshot, ["A","B","C","D"], "B and D look right")
print(generate(msgs))

["B", "D"]


In [5]:
from tqdm import tqdm

def parse_output(text):
    """Return the parsed label, or None if unparseable."""
    text = text.strip()
    # strip code fences if the model added them
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in ["A","B","C","D"] for x in parsed)):
        return parsed
    return None

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

results = []
for ex in tqdm(eval_set):
    msgs = build_messages(fewshot, ex["options"], ex["utterance"])
    raw = generate(msgs)
    parsed = parse_output(raw)
    correct = parsed is not None and labels_equal(parsed, ex["label"])
    results.append({
        "utterance": ex["utterance"],
        "gold": ex["label"],
        "raw": raw,
        "parsed": parsed,
        "correct": correct,
        "valid_format": parsed is not None,
    })

n = len(results)
n_valid = sum(r["valid_format"] for r in results)
n_correct = sum(r["correct"] for r in results)
print(f"format validity: {n_valid}/{n} = {n_valid/n:.1%}")
print(f"exact match:     {n_correct}/{n} = {n_correct/n:.1%}")

100%|██████████| 35/35 [00:19<00:00,  1.76it/s]

format validity: 31/35 = 88.6%
exact match:     26/35 = 74.3%


In [6]:
print("\n=== FAILURES ===")
for r in results:
    if not r["correct"]:
        print(f"utterance: {r['utterance']!r}")
        print(f"  gold:   {r['gold']}")
        print(f"  raw:    {r['raw']!r}")
        print(f"  parsed: {r['parsed']}")
        print()


=== FAILURES ===
utterance: "can't tell"
  gold:   *
  raw:    '[]'
  parsed: []

utterance: "I don't know"
  gold:   *
  raw:    '["*"]'
  parsed: None

utterance: 'this UI is broken'
  gold:   *
  raw:    '[]'
  parsed: []

utterance: 'A great, B meh, C terrible, D good'
  gold:   ['A', 'D']
  raw:    '["A", "C", "D"]'
  parsed: ['A', 'C', 'D']

utterance: 'they all look the same to me'
  gold:   *
  raw:    '[]'
  parsed: []

utterance: 'what time is it?'
  gold:   *
  raw:    '""'
  parsed: None

utterance: 'A is okay but B is closer'
  gold:   ['A', 'B']
  raw:    '["B"]'
  parsed: ['B']

utterance: 'no idea'
  gold:   *
  raw:    '["none of these"]'
  parsed: None

utterance: 'lol'
  gold:   *
  raw:    '""'
  parsed: None



In [7]:
from collections import Counter
def label_kind(l):
    if l == "*": return "star"
    if l == []: return "empty"
    return "list"
print("fewshot:", Counter(label_kind(ex["label"]) for ex in fewshot))
print("eval:   ", Counter(label_kind(ex["label"]) for ex in eval_set))

fewshot: Counter({'list': 12, 'empty': 2, 'star': 1})
eval:    Counter({'list': 25, 'star': 7, 'empty': 3})


In [8]:
def stratify_fewshot(examples, per_category=2, seed=42):
    rng = random.Random(seed)
    star = [e for e in examples if e["label"] == "*"]
    empty = [e for e in examples if e["label"] == []]
    single = [e for e in examples if isinstance(e["label"], list) and len(e["label"]) == 1]
    multi = [e for e in examples if isinstance(e["label"], list) and len(e["label"]) > 1]
    picked = []
    for pool in [star, empty, single, multi]:
        rng.shuffle(pool)
        picked.extend(pool[:per_category*2 if pool is multi else per_category])
    return picked

# rebuild
all_examples = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
random.seed(42); random.shuffle(all_examples)
fewshot = stratify_fewshot(all_examples[:15] + all_examples[35:], per_category=2)  # draw from anywhere
# actually simpler: stratify over the full set, then eval on the rest
fewshot = stratify_fewshot(all_examples, per_category=2)
fewshot_utts = {e["utterance"] for e in fewshot}
eval_set = [e for e in all_examples if e["utterance"] not in fewshot_utts]

print(f"fewshot: {len(fewshot)}, eval: {len(eval_set)}")
print("fewshot label kinds:", Counter(label_kind(e["label"]) for e in fewshot))

fewshot: 10, eval: 40
fewshot label kinds: Counter({'list': 6, 'star': 2, 'empty': 2})
